In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
!ls

Fair-RAG		 evaluation_test.py   rag_eval.sh
X			 final_evaluation.py  simple_evaluation.ipynb
base_eval.py		 final_evaluation.sh  simple_misalignment.py
base_eval.sh		 final_result	      spearman.py
bm25.py			 final_trec_2022      splade.py
contriver.py		 main.ipynb	      stopword-list.txt
correlation_analysis.py  main1.ipynb	      trec-2021
correlation_analysis.sh  main2.ipynb	      trec-2022
evaluation.py		 news		      trec_2021_dataset.ipynb
evaluation.sh		 rag_eval.py


In [144]:
dataset = "trec-2022"
retriever = "splade"
llm = "llama31-8b"
query_topic = "10"
task = "article_generation"
entailment_model = "roberta-large-mnli"
fairness_feature = "general_regions" #"creation_date_category" "years_category" "general_regions"

In [145]:
# Load data
entailment = pd.read_csv(f'./Fair-RAG/entailment/{dataset}/{task}/{query_topic}/{retriever}/{llm}/{entailment_model}/entailment.csv')
delta = pd.read_csv(f'./Fair-RAG/utility_labels/eval_results/{dataset}/{task}/{query_topic}/{retriever}/{llm}/5_delta.tsv', delimiter="\t")
if "trec" in dataset:
    corpus = pd.read_csv(f'./{dataset}/document_features.csv')
else:
    corpus = pd.read_csv(f'./{dataset}/{dataset}_collection.csv')

In [146]:
delta.columns

Index(['qid', 'pid', 'augment_score', 'baseline_score', 'delta'], dtype='object')

In [147]:
# import pandas as pd
# import numpy as np

# # --- Rename columns for consistency ---
# entailment.rename(columns={"doc_id": "qid", "profile_doc_id": "pid", "score": "attention"}, inplace=True)

# # --- Merge on query id (qid) and document id (pid) ---
# merged_df = pd.merge(entailment, delta[["qid", "pid", "delta"]], on=["qid", "pid"], how="inner")

# # --- Group by query ---
# grouped = merged_df.groupby("qid")

# attention_matrix = []
# utility_matrix = []
# query_ids = []

# for qid, group in grouped:
#     group_sorted = group.sort_values("pid")  # Ensure consistent order if needed

#     if len(group_sorted) != 10:
#         print(f"Skipping query {qid}, expected 10 documents but got {len(group_sorted)}")
#         continue

#     attention_row = group_sorted["attention"].values
#     utility_row = group_sorted["delta"].values

#     attention_matrix.append(attention_row)
#     utility_matrix.append(utility_row)
#     query_ids.append(qid)

# # Convert to NumPy arrays
# attention_matrix = np.array(attention_matrix)
# utility_matrix = np.array(utility_matrix)

# print("attention_matrix shape:", attention_matrix.shape)
# print("utility_matrix shape:", utility_matrix.shape)


In [148]:
# non_useful_attention = np.sum((utility_matrix <= 0) * attention_matrix, axis=1)
# useful_attention = np.sum((utility_matrix > 0) * attention_matrix, axis=1)

In [149]:
# non_useful_attention, useful_attention

In [150]:
# import numpy as np
# performance = np.loadtxt(f"./Fair-RAG/utility_labels/eval_results/{dataset}/{query_topic}/{retriever}/{llm}/5_per_ranking_rouge.txt", delimiter=",", usecols=1)

In [151]:
# diff = useful_attention - non_useful_attention

# # Create binary result: 1 if diff > 0, else 0
# binary_result = (diff > 0).astype(int)

In [152]:
# from scipy.stats import pearsonr, spearmanr

# pearson_corr, _ = pearsonr(diff, performance)
# spearman_corr, _ = spearmanr(diff, performance)

# print(f"Pearson: {pearson_corr:.4f}, Spearman: {spearman_corr:.4f}")

In [153]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# sns.scatterplot(x=non_useful_attention, y=performance)
# plt.xlabel("Attention on Non-Useful Docs")
# plt.ylabel("LLM Performance (e.g., ROUGE-L)")
# plt.title("Impact of Non-Useful Attention on Performance")
# plt.grid(True)
# plt.show()

In [154]:
# Step 1: Label usage from entailment (usage = 1 if score > 0.5)
entailment['usage'] = entailment['score'] > 0.5

# Step 2: Label relevance from delta (relevant = 1 if delta > 0)
delta['relevant'] = delta['delta'] > 0

# Step 3: Merge everything into a single DataFrame
merged = entailment.merge(delta, left_on=['doc_id', 'profile_doc_id'], right_on=['qid', 'pid'])
# fairness_feature = "creation_date_category" #"creation_date_category" "years_category"
merged = merged.merge(corpus[['docno', fairness_feature]], left_on='profile_doc_id', right_on='docno')

# Step 4: Compute per-query bias score
all_groups = merged[fairness_feature].unique().tolist()

In [155]:
merged

,doc_id,profile_doc_id,score,usage,qid,pid,augment_score,baseline_score,delta,relevant,docno,general_regions
0,63769611,29337192,0.582827,True,63769611,29337192,0.179837,0.223022,-0.043185,False,29337192,['None']
1,63769611,38030472,0.361883,False,63769611,38030472,0.160256,0.223022,-0.062765,False,38030472,['Asia']
2,63769611,48719498,0.102087,False,63769611,48719498,0.179949,0.223022,-0.043073,False,48719498,['Asia']
3,63769611,40352523,0.544359,True,63769611,40352523,0.208633,0.223022,-0.014388,False,40352523,"['Asia', 'Asia', 'Europe']"
4,63769611,34009775,0.180907,False,63769611,34009775,0.184397,0.223022,-0.038624,False,34009775,['Asia']
...,...,...,...,...,...,...,...,...,...,...,...,...
1395,34488061,1906607,0.360703,False,34488061,1906607,0.141935,0.237624,-0.095688,False,1906607,['Americas']
1396,34488061,11580179,0.098320,False,34488061,11580179,0.039604,0.237624,-0.198020,False,11580179,['Americas']
1397,34488061,58577780,0.086801,False,34488061,58577780,0.164384,0.237624,-0.073240,False,58577780,['None']
1398,34488061,2241685,0.146384,False,34488061,2241685,0.271777,0.237624,0.034153,True,2241685,['Europe']


In [156]:
all_groups

["['None']",
 "['Asia']",
 "['Asia', 'Asia', 'Europe']",
 "['Europe']",
 "['Antarctica']",
 "['Americas']",
 "['Antarctica', 'Europe']",
 "['Oceania']",
 "['Africa']",
 "['Asia', 'Americas']",
 "['Europe', 'Europe']",
 "['Europe', 'Asia']",
 "['Americas', 'Americas']",
 "['Europe', 'Americas']"]

In [157]:
def compute_alignment_misalignment_ratios(merged_df):
    """
    Computes alignment and misalignment ratios for each query.

    Alignment: (used == relevant)
    Misalignment: (used != relevant)
    """
    from collections import defaultdict

    alignment_stats = defaultdict(dict)

    grouped = merged_df.groupby("qid")

    for qid, group in grouped:
        used = group["usage"].values
        relevant = group["relevant"].values

        total = len(used)
        if total < 10:
            print("here")
        aligned = np.sum(used == relevant)
        misaligned = total - aligned

        alignment_stats[qid]["alignment_ratio"] = aligned / total
        alignment_stats[qid]["misalignment_ratio"] = misaligned / total

    alignment_df = pd.DataFrame.from_dict(alignment_stats, orient="index").reset_index()
    alignment_df.rename(columns={"index": "qid"}, inplace=True)
    return alignment_df

def compute_alignment_misalignment_ratios_per_group(merged_df):
    """
    Computes alignment and misalignment ratios for each query and fairness group.

    Alignment: usage == relevant
    Misalignment: usage != relevant
    """
    from collections import defaultdict

    alignment_stats = []

    grouped = merged_df.groupby(["qid", fairness_feature])  # Group by query and fairness group

    for (qid, bias), group in grouped:
        used = group["usage"].values
        relevant = group["relevant"].values

        total = len(used)
        if total == 0:
            alignment_stats.append({
            "qid": qid,
            fairness_feature: bias,
            "alignment_ratio": 0,
            "misalignment_ratio": 0,
            "total_docs": total,
            "aligned": 0,
            "misaligned": 0
            })
            print("!!!!!!!!!!!!!")
            continue  # skip empty groups to avoid division by zero

        aligned = np.sum(used == relevant)
        misaligned = total - aligned

        alignment_stats.append({
            "qid": qid,
            fairness_feature: bias,
            "alignment_ratio": aligned / total,
            "misalignment_ratio": misaligned / total,
            "total_docs": total,
            "aligned": aligned,
            "misaligned": misaligned
        })

    alignment_df = pd.DataFrame(alignment_stats)
    return alignment_df

def average_alignment_misalignment_per_query(alignment_group_df):
    """
    Computes the average alignment and misalignment ratios for each query,
    by averaging across all fairness groups.
    """
    avg_df = alignment_group_df.groupby("qid")[["alignment_ratio", "misalignment_ratio"]].mean().reset_index()
    return avg_df


def compute_bias_score(df):
    group_to_idx = {g: i for i, g in enumerate(all_groups)}
    rel_dist = np.zeros(len(all_groups))
    use_dist = np.zeros(len(all_groups))

    for _, row in df.iterrows():
        idx = group_to_idx[row[fairness_feature]]
        rel_dist[idx] += int(row['relevant'])
        use_dist[idx] += int(row['usage'])

    if rel_dist.sum() > 0:
        rel_dist /= rel_dist.sum()
    if use_dist.sum() > 0:
        use_dist /= use_dist.sum()

    return np.sum((rel_dist - use_dist) ** 2)
    
def per_group_precision(df):
    group_correct = defaultdict(int)
    group_total = defaultdict(int)
    for _, row in df.iterrows():
        if row['usage']:
            group_total[row[fairness_feature]] += 1
            if row['relevant']:
                group_correct[row[fairness_feature]] += 1
    return {g: group_correct[g] / group_total[g] if group_total[g] > 0 else None for g in group_total}
    
def per_group_recall(df):
    group_relevant = defaultdict(int)
    group_used_relevant = defaultdict(int)
    for _, row in df.iterrows():
        if row['relevant']:
            group_relevant[row[fairness_feature]] += 1
            if row['usage']:
                group_used_relevant[row[fairness_feature]] += 1
    return {g: group_used_relevant[g] / group_relevant[g] if group_relevant[g] > 0 else None for g in group_relevant}
    

In [158]:
alignment_df = compute_alignment_misalignment_ratios(merged)
print(alignment_df)

          qid  alignment_ratio  misalignment_ratio
0       61768              0.7                 0.3
1      214449              0.5                 0.5
2      214475              0.5                 0.5
3      384817              0.6                 0.4
4      644472              0.7                 0.3
..        ...              ...                 ...
135  68191430              0.7                 0.3
136  68606566              0.4                 0.6
137  69115086              0.4                 0.6
138  69303097              0.4                 0.6
139  69817945              0.0                 1.0

[140 rows x 3 columns]


In [159]:
alignment_group_df = compute_alignment_misalignment_ratios_per_group(merged)
alignment_group_df.head(30)

,qid,general_regions,alignment_ratio,misalignment_ratio,total_docs,aligned,misaligned
0,61768,['Americas'],1.000000,0.000000,2,2,0
1,61768,['Asia'],0.000000,1.000000,2,0,2
2,61768,['Europe'],0.500000,0.500000,2,1,1
3,61768,['None'],1.000000,0.000000,4,4,0
4,214449,['Africa'],0.500000,0.500000,6,3,3
5,214449,['Americas'],0.000000,1.000000,1,0,1
6,214449,['None'],0.666667,0.333333,3,2,1
7,214475,['Africa'],0.428571,0.571429,7,3,4
8,214475,['None'],0.666667,0.333333,3,2,1
9,384817,['Antarctica'],0.600000,0.400000,10,6,4


In [160]:
# Compute per-query average alignment/misalignment
avg_alignment_df = average_alignment_misalignment_per_query(alignment_group_df)
avg_alignment_df["fair"] = (avg_alignment_df["alignment_ratio"] > avg_alignment_df["misalignment_ratio"]).astype(int)
avg_alignment_df["distance"] = (avg_alignment_df["alignment_ratio"] - avg_alignment_df["misalignment_ratio"])

# Preview results
avg_alignment_df.head(10)

,qid,alignment_ratio,misalignment_ratio,fair,distance
0,61768,0.625000,0.375000,1,0.250000
1,214449,0.388889,0.611111,0,-0.222222
2,214475,0.547619,0.452381,1,0.095238
3,384817,0.600000,0.400000,1,0.200000
4,644472,0.604167,0.395833,1,0.208333
5,644586,0.066667,0.933333,0,-0.866667
6,723546,0.541667,0.458333,1,0.083333
7,899456,0.583333,0.416667,1,0.166667
8,981993,0.200000,0.800000,0,-0.600000
9,1112156,0.722222,0.277778,1,0.444444


In [161]:
import numpy as np
from scipy.stats import pearsonr, spearmanr, kendalltau

performance = np.loadtxt(f"./Fair-RAG/utility_labels/eval_results/{dataset}/{task}/{query_topic}/{retriever}/{llm}/5_per_ranking_rouge.txt", delimiter=",", usecols=1)
unfairness = np.array(avg_alignment_df["distance"].tolist())

# pearson_corr, pearson_p = pearsonr(performance, unfairness)
# print(f"Pearson correlation: {pearson_corr:.4f} (p-value: {pearson_p:.4e})")
# Spearman correlation (non-parametric)
spearman_corr, spearman_p = spearmanr(unfairness, performance)
print(f"Spearman correlation: {spearman_corr:.4f} (p-value: {spearman_p:.4e})")

tau, p_value = kendalltau(unfairness, performance)
print(f"Kendaltau correlation: {tau:.4f} (p-value: {p_value:.4e})")

Spearman correlation: -0.0880 (p-value: 3.0112e-01)
Kendaltau correlation: -0.0612 (p-value: 2.8530e-01)


In [65]:
bias_scores = merged.groupby('doc_id').apply(compute_bias_score).reset_index()
bias_scores.columns = ['query_id', 'bias_score']

/tmp/ipykernel_3389/2262512103.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bias_scores = merged.groupby('doc_id').apply(compute_bias_score).reset_index()


In [66]:
bias_scores

,query_id,bias_score
0,62905,0.625000
1,75015,0.000000
2,115380,0.000000
3,120188,1.000000
4,121201,0.722222
...,...,...
102,66362218,1.000000
103,66363156,0.375000
104,66380427,1.000000
105,67761800,0.000000


In [67]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

performance = np.loadtxt(f"./Fair-RAG/utility_labels/eval_results/{dataset}/{retriever}/{llm}/5_per_ranking_rouge.txt", delimiter=",", usecols=1)
unfairness = np.array(bias_scores["bias_score"].tolist())

pearson_corr, pearson_p = pearsonr(unfairness, performance)
print(f"Pearson correlation: {pearson_corr:.4f} (p-value: {pearson_p:.4e})")
# Spearman correlation (non-parametric)
spearman_corr, spearman_p = spearmanr(unfairness, performance)
print(f"Spearman correlation: {spearman_corr:.4f} (p-value: {spearman_p:.4e})")

Pearson correlation: -0.0452 (p-value: 6.4423e-01)
Spearman correlation: -0.0368 (p-value: 7.0672e-01)


In [68]:
precision_per_group = per_group_precision(merged)

In [69]:
precision_per_group

{'Low': 0.2898550724637681,
 'Medium-Low': 0.3333333333333333,
 'Medium-High': 0.30434782608695654,
 'High': 0.5652173913043478}

In [70]:
recall_per_group = per_group_recall(merged)

In [71]:
recall_per_group

{'Low': 0.5161290322580645,
 'Medium-Low': 0.2909090909090909,
 'Medium-High': 0.2,
 'High': 0.3023255813953488}

In [72]:
bias_scores.head(), precision_per_group, recall_per_group

(   query_id  bias_score
 0     62905    0.625000
 1     75015    0.000000
 2    115380    0.000000
 3    120188    1.000000
 4    121201    0.722222,
 {'Low': 0.2898550724637681,
  'Medium-Low': 0.3333333333333333,
  'Medium-High': 0.30434782608695654,
  'High': 0.5652173913043478},
 {'Low': 0.5161290322580645,
  'Medium-Low': 0.2909090909090909,
  'Medium-High': 0.2,
  'High': 0.3023255813953488})